# Notebook 04 — Full End-to-End Pipeline

Reproduces exactly what the web app does, step by step:

```
Image → Block Detection → TrOCR → Question Mapping → Grading → Results
```

No browser needed. Same functions, same models, same output.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join('..', 'engine'))

import cv2
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np

from sheet_analyzer import analyze_sheet, analyze_pil_sheet, locate_answer_blocks, draw_annotations
from answer_scorer  import evaluate_answer, evaluate_multiple, run_ablation, GradingMode

print('All imports OK.')

## Step 0 — Configuration

Set your image path and teacher answers here. Everything else runs automatically.

In [ ]:
# ── EDIT THESE ───────────────────────────────────────────────────────────────

IMAGE_PATH = "../sheet.png"   # path to your answer sheet image (PNG / JPG)

# One string per question. Use a single-item list for a whole-sheet evaluation.
TEACHER_ANSWERS = [
    "Photosynthesis is the process by which plants use sunlight, water, and carbon dioxide to produce glucose and oxygen.",
    # "Newton's second law states that force equals mass times acceleration, F = ma.",
    # add more answers if your sheet has multiple questions
]

GRADING_MODE = GradingMode.HYBRID   # GradingMode.ML | GradingMode.DL | GradingMode.HYBRID

# ─────────────────────────────────────────────────────────────────────────────
print(f'Image      : {IMAGE_PATH}')
print(f'Questions  : {len(TEACHER_ANSWERS)}')
print(f'Mode       : {GRADING_MODE.value}')

## Step 1 — Load & Display the Image

In [ ]:
assert os.path.exists(IMAGE_PATH), f"Image not found: {IMAGE_PATH}"

img_bgr = cv2.imread(IMAGE_PATH)
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(10, 6))
plt.imshow(img_rgb)
plt.title('Original Answer Sheet')
plt.axis('off')
plt.tight_layout()
plt.show()

print(f'Image size: {img_bgr.shape[1]}w x {img_bgr.shape[0]}h px')

## Step 2 — Detect Answer Blocks

In [ ]:
blocks = locate_answer_blocks(img_bgr)
annotated_bgr = draw_annotations(img_bgr, blocks)

plt.figure(figsize=(10, 6))
plt.imshow(cv2.cvtColor(annotated_bgr, cv2.COLOR_BGR2RGB))
plt.title(f'Detected Answer Blocks ({len(blocks)} found)')
plt.axis('off')
plt.tight_layout()
plt.show()

print(f'Detected {len(blocks)} blocks:')
for i, (x1, y1, x2, y2) in enumerate(blocks):
    print(f'  Block {i+1}: x=[{x1},{x2}]  y=[{y1},{y2}]  size={x2-x1}×{y2-y1}px')

## Step 3 — TrOCR: Read Text from Each Block

In [ ]:
from sheet_analyzer import trocr_read_block

block_texts = []
for i, (x1, y1, x2, y2) in enumerate(blocks):
    print(f'OCR-ing block {i+1}/{len(blocks)} ...', end=' ', flush=True)
    text = trocr_read_block(img_bgr, x1, y1, x2, y2)
    block_texts.append(text)
    print(f'done → "{text[:60]}{'...' if len(text)>60 else ''}"')

print('\n── Extracted Texts ──')
for i, t in enumerate(block_texts):
    print(f'\n[Block {i+1}]\n{t}')

## Step 4 — Map Blocks to Questions

In [ ]:
from sheet_analyzer import map_blocks_to_questions

question_map = map_blocks_to_questions(block_texts)

print('Question Map:')
for q_num, text in question_map.items():
    print(f'  Q{q_num}: "{text[:80]}{'...' if len(text)>80 else ''}"')

## Step 5 — Grade

In [ ]:
if len(TEACHER_ANSWERS) > 1:
    result = evaluate_multiple(question_map, TEACHER_ANSWERS, mode=GRADING_MODE)

    print(f'\n═══ RESULTS (mode={GRADING_MODE.value}) ═══')
    print(f'Combined: {result["combined_marks"]}/{result["combined_total"]}  →  {result["combined_grade"]}')
    print()
    for q in result['per_question']:
        print(f'  Q{q["question"]}: {q["marks"]}/{q["total"]}  {q["grade"]}  (similarity={q["similarity"]})')
        print(f'    Matched : {q["matched"]}')
        print(f'    Missed  : {q["missed"]}')

else:
    student_text = "\n".join(block_texts)
    result = evaluate_answer(student_text, TEACHER_ANSWERS[0], mode=GRADING_MODE)

    print(f'\n═══ RESULTS (mode={GRADING_MODE.value}) ═══')
    print(f'Score     : {result["marks"]}/{result["total"]}')
    print(f'Grade     : {result["grade"]}')
    print(f'Similarity: {result["similarity"]}')
    print(f'Matched   : {result["matched"]}')
    print(f'Missed    : {result["missed"]}')

## Step 6 — Ablation: Compare All Three Models on the Same Input

In [ ]:
import pandas as pd

student_text = "\n".join(block_texts)
teacher_text = TEACHER_ANSWERS[0]

ablation = run_ablation(student_text, teacher_text)

arch = {
    'ml'    : 'TF-IDF bigrams + Jaccard',
    'dl'    : 'RoBERTa cross-encoder + SBERT',
    'hybrid': 'Cross-encoder + BiLSTM + TF-IDF',
}
rows = [
    {
        'Model'       : {'ml':'A — ML','dl':'B — DL','hybrid':'C — Hybrid'}[m],
        'Architecture': arch[m],
        'Similarity'  : f"{d['similarity']:.3f}",
        'Marks'       : f"{d['marks']}/10",
        'Grade'       : d['grade'],
    }
    for m, d in ablation.items()
]
df = pd.DataFrame(rows).set_index('Model')
print(df.to_string())

## Step 7 — Visual Summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: annotated sheet
axes[0].imshow(cv2.cvtColor(annotated_bgr, cv2.COLOR_BGR2RGB))
axes[0].set_title('Detected Blocks')
axes[0].axis('off')

# Right: ablation bar chart
models = [r['Model'] for r in rows]
sims   = [float(r['Similarity']) for r in rows]
colors = ['#f59e0b', '#3b82f6', '#10b981']
bars   = axes[1].bar(models, sims, color=colors, alpha=0.85, width=0.5)
axes[1].set_ylim(0, 1.05)
axes[1].set_ylabel('Similarity Score')
axes[1].set_title('Ablation — All Three Models')
axes[1].axhline(0.7, color='gray', linestyle='--', linewidth=0.8, alpha=0.6)
for bar, val in zip(bars, sims):
    axes[1].text(bar.get_x() + bar.get_width()/2, val + 0.02,
                 f'{val:.3f}', ha='center', va='bottom', fontsize=11)

plt.tight_layout()
plt.show()